In [ ]:
import copy
import sys
# from FocalLoss import FocalLoss
import pickle
import matplotlib.pyplot as plt
import torch.utils.data as Data
import numpy as np
from numpy import dtype
from sympy.physics.units import second
from torch.profiler import schedule

# from models import DIN
# from hidden_detach import BlockRNN
from models_revin import BlockRNN, count_parameters
from torch import nn
import torch
from tqdm import tqdm
import pandas as pd
from data_xrd_elements import XRDDataset,muti_XRDDataset
from FocalLoss import FocalLoss

In [ ]:
bs = 512  # 64 # 32 #
dvsid = [0, 1, 2, 3]
news = "-muti_pg250616"

In [ ]:
def collate_fn(batch):
    features, targets,pre, objects = zip(*batch)  # 解包batch
    features = torch.stack(features)  # 把features变成一个batch的张量
    targets = torch.stack(targets)  # 标签也堆叠成张量
    pres = torch.stack(pre)
    return features, targets,pres, objects

In [ ]:
def train(model, train_dataloader, val_dataloader, model_name,weights, epochs=10):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    step = 1e-5
    optimizer = torch.optim.Adam(model.parameters(), lr=step,weight_decay=1e-4)  #Adam(model.parameters(), lr=3e-4)#  0.0003     4e-3
    # scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.8)
    num_max = max(weights)
    weights = [num_max / i for i in weights]
    weights = torch.FloatTensor(weights).cuda()
    criterion = nn.CrossEntropyLoss(weight=weights)
    model = nn.DataParallel(model, output_device=dvsid).to(device)  #model #
    # model  = model.to(device)
    lossOfepochInTrain = []
    accOfepochInTrain = []
    lossOfepochInVal = []
    accOfepochInVal = []
    nanLossCount = 0
    early_stop = 20
    with torch.no_grad():
        model.eval()
        val_loss = 0
        val_correct = 0
        val_num = 0
        for data, target,pre_, _ in tqdm(val_dataloader):
            data, target,pre_ = data.to(device), target.to(device),pre_.to(device)
            data = data.unsqueeze(1)
            output = model(data,pre_)
            pre = torch.argmax(output, dim=1)
            loss = criterion(output, target)
            val_loss += loss.item() * data.size(0)
            val_correct += torch.sum(pre == target.data)
            val_num += data.size(0)
    min_val_loss = val_loss / val_num
    best_acc = val_correct.double().item() / val_num
    tqdm.write(f"min_val_loss:{min_val_loss:.4f} best_acc:{best_acc:.4f}")
    for epoch in range(epochs):
        train_loss = 0
        train_correct = 0
        train_num = 0
        val_loss = 0
        val_correct = 0
        val_num = 0
        model.train()
        for data, target,pre_, _ in tqdm(train_dataloader, desc=f"epoch:{epoch + 1}/{epochs}", unit="batch",
                                    file=sys.stdout):
            data, target,pre_ = data.to(device), target.to(device),pre_.to(device)  # .cuda() data.to(device), target.to(device)
            optimizer.zero_grad()
            # print(hidden.size())
            # print(hidden.detach().size())
            # print(data.shape)
            data = data.unsqueeze(1)
            output = model(data,pre_)

            # hidden = hidden.data
            pre = torch.argmax(output, dim=1)
            loss = criterion(output, target)
            loss.backward()
            # nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            # scheduler.step()
            train_loss += loss.item() * data.size(0)
            train_correct += torch.sum(pre == target.data)
            train_num += data.size(0)

        lossOfepochInTrain.append(train_loss / train_num)
        accOfepochInTrain.append(train_correct.double().item() / train_num)
        tqdm.write(
            f"epoch:{epoch + 1}/{epochs} train_loss:{train_loss / train_num:.4f} train_acc:{train_correct / train_num:.4f}")
        if np.isnan(train_loss):
            nanLossCount += 1

        # 在验证集上验证
        with torch.no_grad():
            model.eval()
            hidden = None  #torch.zeros(hid0 * n_layers, bs, hs)
            for data, target,pre_, _ in tqdm(val_dataloader, desc=f"epoch:{epoch + 1}/{epochs}", unit="batch",
                                        file=sys.stdout):
                data, target,pre_ = data.to(device), target.to(device),pre_.to(device)  # .cuda() data.to(device), target.to(device)
                data = data.unsqueeze(1)
                output = model(data,pre_)
                pre = torch.argmax(output, dim=1)
                loss = criterion(output, target)
                val_loss += loss.item() * data.size(0)
                val_correct += torch.sum(pre == target.data)
                val_num += data.size(0)
        lossOfepochInVal.append(val_loss / val_num)
        accOfepochInVal.append(val_correct.double().item() / val_num)
        epoch_infos = f"epoch:{epoch + 1}/{epochs} val_loss:{val_loss / val_num:.4f} val_acc:{val_correct / val_num:.4f}"
        if np.isnan(val_loss):
            nanLossCount += 1
        if nanLossCount >= 3:
            raise (Exception("There will not be a fourth time for nan loss."))
        val_loss_diff = lossOfepochInVal[-1] - min_val_loss

        if accOfepochInVal[-1] > best_acc:
            tqdm.write(epoch_infos + " ... ModelSaving")
            best_acc = accOfepochInVal[-1]
            best_model_wts = copy.deepcopy(model.module.state_dict()) if isinstance(model,
                                                                                    nn.DataParallel) else copy.deepcopy(
                model.state_dict())
            torch.save(best_model_wts, f'{model_name}{news}.pth')
            # early_stop = 20

        else:
            # early_stop -= 1
            # if early_stop == 0:
            #     step = step / 2
            #     optimizer = torch.optim.Adam(model.parameters(), lr=step,weight_decay=1e-4)
            #     early_stop = 20
            tqdm.write(epoch_infos)

    # 训练记录
    train_process = pd.DataFrame(data={'epoch': range(1, epochs + 1),
                                       'loss in train': lossOfepochInTrain,
                                       'acc in train': accOfepochInTrain,
                                       'loss in val': lossOfepochInVal,
                                       'acc in val': accOfepochInVal, })
    train_process.to_csv(f'{model_name}-{news}.csv', index=False)

    return train_process


In [ ]:
from resnet import RCNet

model = RCNet(
    in_channels=1,
    base_filters=16,
    kernel_size=15,
    stride=2,
    groups=1,
    n_block=16,
    n_classes=32,
    downsample_gap=2,
    increasefilter_gap=4,
    use_bn=True,
    use_do=True,
    num_heads=2
)


In [ ]:
train_root = "3attention/train_data_3attention_260113.npy"
val_root = "3attention/val_data_3attention_260113.npy"
test_root = "3attention/test_data_3attention_260113.npy"
train_data = muti_XRDDataset(train_root, cls='point_group')
val_data =  muti_XRDDataset(val_root, cls='point_group')
test_data = muti_XRDDataset(test_root, cls='point_group')
# print(dataset.get_label_map())
train_loader = Data.DataLoader(dataset=train_data, batch_size=bs, shuffle=True,num_workers=8, drop_last=True,
                               collate_fn=collate_fn)
val_loader = Data.DataLoader(dataset=val_data, batch_size=bs, shuffle=True, num_workers=8,drop_last=True,
                             collate_fn=collate_fn)
test_loader = Data.DataLoader(dataset=test_data, batch_size=bs, shuffle=True, num_workers=8,drop_last=True,
                              collate_fn=collate_fn)

In [ ]:
data = np.load(train_root, allow_pickle=True)
data_pd = pd.DataFrame(data)
point_group = ['1', '-1', '2', 'm', '2/m', '222', 'mm2', 'mmm', '4', '-4', '4/m', '422', '4mm', '-42m', '4/mmm', '3', '-3', '32', '3m', '-3m', '6', '-6', '6/m', '622', '6mm', '-6m2', '6/mmm', '23', 'm-3', '432', '-43m', 'm-3m']
weights = data_pd.value_counts(5).reindex(point_group).to_numpy()

In [ ]:
pd.DataFrame(np.load(test_root, allow_pickle=True))

In [ ]:
train(model, train_loader, val_loader, "resnet",weights,epochs= 800)

In [ ]:
def test(model, val_dataloader):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    # optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()  #FocalLoss(gamma=2)
    dvsid = [0, 1, 2, 3]
    model = nn.DataParallel(model, output_device=dvsid).to(device)  # model.to(device) #
    # lossOfepochInTrain = []
    # accOfepochInTrain = []
    lossOfepochInVal = []
    accOfepochInVal = []
    conf_mat_data = [[[] for i in range(32)] for i in range(32)]
    first_acc_mat = [[[] for i in range(32)] for i in range(32)]
    second_acc_mat = [[[] for i in range(32)] for i in range(32)]
    conf_mat = np.zeros((32, 32))
    val_loss = 0
    val_correct = 0
    val_num = 0
    # best_acc = 0.0
    # 在验证集上验证
    with torch.no_grad():
        model.eval()
        for data, target, pre_,xrd in tqdm(val_dataloader):
            data, target ,pre_= data.to(device), target.to(device),pre_.to(device)  #data.to(device), target.to(device)
            data = data.unsqueeze(1)
            output = model(data,pre_)
            pre = torch.argmax(output, dim=1)

            loss = criterion(output, target)
            val_loss += loss.item() * data.size(0)
            val_correct += torch.sum(pre == target.data)
            val_num += data.size(0)
        lossOfepochInVal.append(val_loss / val_num)
        accOfepochInVal.append(val_correct.double().item() / val_num)
    tqdm.write(f"test_loss:{val_loss / val_num:.4f} test_acc:{val_correct / val_num:.4f}")
    return conf_mat, conf_mat_data, first_acc_mat, second_acc_mat

In [ ]:
conf_mat, conf_mat_data, first_mat, second_mat = test(model, train_loader)